# Compile the test prompts into one table (Colab)

Joins the generated dbs in `My Drive/ez-penta/` into one table, `answercarefully_test_prompts.sqlite3`, saved back to Drive:

| Columns | Taken from |
|---|---|
| `source_id`, `source_split` | row key, shared by every db |
| `text`, `text_en` | original Japanese and English prompts (in every db; checked to match) |
| `fc_ja`, `fc_en` | `answercarefully_qwen_fc.sqlite3` |
| `rp_ja`, `rp_en` | `answercarefully_qwen_rp.sqlite3` |
| `rs_ja`, `rs_en` | `answercarefully_qwen_rs.sqlite3` |
| `flake_every10_*`, `flake_quartile_*` (8) | `answercarefully_simple.sqlite3` |
| `output` | expected Japanese reference answer (in every db; checked to match) |

Each Qwen db's own column names (for example `Fiction_ja`) are read from its `generation_info` table, so a different `RUN_NAME` doesn't matter. No GPU needed: any Colab server works.

In [ ]:
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/ez-penta")
QWEN_DBS = {
    "fc": DRIVE_DIR / "answercarefully_qwen_fc.sqlite3",
    "rp": DRIVE_DIR / "answercarefully_qwen_rp.sqlite3",
    "rs": DRIVE_DIR / "answercarefully_qwen_rs.sqlite3",
}
SIMPLE_DB = DRIVE_DIR / "answercarefully_simple.sqlite3"
DB_OUT = DRIVE_DIR / "answercarefully_test_prompts.sqlite3"
WORK_DIR = Path("/content/work")

KEY = ["source_id", "source_split"]
SHARED = ["text", "text_en", "output"]  # identical in every db, so taken once

## 1. Download the dbs from Google Drive

In [ ]:
import shutil

if not DRIVE_DIR.exists():
    from google.colab import drive
    drive.mount("/content/drive")
missing = [path.name for path in [*QWEN_DBS.values(), SIMPLE_DB] if not path.exists()]
if missing:
    raise FileNotFoundError(f"Not in {DRIVE_DIR}: {missing}. Run the notebooks that make them first.")

# SQLite is unreliable directly on the Drive mount, so read local copies.
WORK_DIR.mkdir(parents=True, exist_ok=True)
for path in [*QWEN_DBS.values(), SIMPLE_DB]:
    shutil.copy(path, WORK_DIR / path.name)
    print(f"Copied {path} -> {WORK_DIR / path.name}")

In [ ]:
import sqlite3

import pandas as pd


def read_db(path):
    connection = sqlite3.connect(WORK_DIR / path.name)
    records = pd.read_sql_query("SELECT * FROM records", connection)
    info = pd.read_sql_query("SELECT * FROM generation_info", connection)
    connection.close()
    return records, info


simple, simple_info = read_db(SIMPLE_DB)
FLAKE_COLUMNS = [column for column in simple.columns if column.startswith("flake_")]
base = simple[KEY + SHARED + FLAKE_COLUMNS]

sources = [{"db": SIMPLE_DB.name, "columns": ", ".join(FLAKE_COLUMNS), "renamed_from": ", ".join(FLAKE_COLUMNS)}]
qwen_columns = {}
for strategy, path in QWEN_DBS.items():
    records, info = read_db(path)
    # The save cell records the output columns, e.g. "Fiction_ja, Fiction_en".
    generated = info["columns"].iloc[0].split(", ")
    rename = {column: f"{strategy}_{column.rsplit('_', 1)[1]}" for column in generated}
    if sorted(rename.values()) != [f"{strategy}_en", f"{strategy}_ja"]:
        raise ValueError(f"{path.name}: expected one _ja and one _en column, found {generated}")
    merged = base.merge(records[KEY + SHARED + generated].rename(columns=rename), on=KEY, how="left",
                        suffixes=("", f"_{strategy}"), validate="one_to_one")
    for column in SHARED:
        # Same rows from the same original db: anything else means a db was built from different data.
        mismatched = (merged[column] != merged[f"{column}_{strategy}"]) & merged[f"{column}_{strategy}"].notna()
        if mismatched.any():
            raise ValueError(f"{path.name}: {column} differs from {SIMPLE_DB.name} in {mismatched.sum()} rows")
    base = merged.drop(columns=[f"{column}_{strategy}" for column in SHARED])
    qwen_columns[strategy] = [f"{strategy}_ja", f"{strategy}_en"]
    sources.append({"db": path.name, "columns": ", ".join(qwen_columns[strategy]), "renamed_from": ", ".join(generated)})
    print(f"{path.name}: {generated} -> {qwen_columns[strategy]}")

QWEN_COLUMNS = [column for columns in qwen_columns.values() for column in columns]
ORDER = KEY + ["text", "text_en"] + QWEN_COLUMNS + FLAKE_COLUMNS + ["output"]
test_prompts = base[ORDER]
print(f"{len(test_prompts)} rows, {len(ORDER)} columns")

## 2. Check coverage

The Qwen columns are empty for rows a notebook hasn't processed yet (its `LIMIT`), so check how many rows have everything.

In [ ]:
PROMPT_COLUMNS = QWEN_COLUMNS + FLAKE_COLUMNS
coverage = test_prompts[PROMPT_COLUMNS].notna().sum().rename("filled").to_frame()
coverage["missing"] = len(test_prompts) - coverage["filled"]
display(coverage)
complete = test_prompts[PROMPT_COLUMNS].notna().all(axis=1)
print(f"{complete.sum()} of {len(test_prompts)} rows have all {len(PROMPT_COLUMNS)} test prompts")

In [ ]:
# One row with every column, read top to bottom.
pd.set_option("display.max_colwidth", None)
row = complete.idxmax() if complete.any() else 0
test_prompts.loc[row].to_frame(f"row {row}")

## 3. Save as a new SQLite db (and copy it to Drive)

In [ ]:
from datetime import datetime, timezone

LOCAL_OUT = WORK_DIR / DB_OUT.name
LOCAL_OUT.unlink(missing_ok=True)
connection = sqlite3.connect(LOCAL_OUT)
definitions = [f'"{column}" TEXT' + (" NOT NULL" if column in KEY + ["text", "output"] else "") for column in ORDER]
rows = ([None if pd.isna(value) else value for value in row] for row in test_prompts.itertuples(index=False))
with connection:
    connection.execute(f"CREATE TABLE records ({', '.join(definitions)}, PRIMARY KEY ({', '.join(KEY)}))")
    connection.executemany(f"INSERT INTO records ({', '.join(ORDER)}) VALUES ({', '.join('?' * len(ORDER))})", rows)
    connection.execute("CREATE TABLE compile_info (db TEXT NOT NULL, columns TEXT NOT NULL, renamed_from TEXT NOT NULL, compiled_at TEXT NOT NULL)")
    compiled_at = datetime.now(timezone.utc).isoformat()
    connection.executemany("INSERT INTO compile_info VALUES (?, ?, ?, ?)",
                           [(source["db"], source["columns"], source["renamed_from"], compiled_at) for source in sources])
print([name for _, name, *_ in connection.execute("PRAGMA table_info(records)")])
connection.close()

shutil.copy(LOCAL_OUT, DB_OUT)
print(f"Saved {DB_OUT}")